# Thermal Paths as Structure — ESC Cold-Plate & Vented Battery Bay
### Electric V-BAT-Like Tail-Sitter (EDF) — Conceptual Design Study

---

## Purpose

The airframe carries two continuous heat sources in hover — the **ESC**
(switching/conduction loss) and the **battery** (pack IR loss) — and until
now had no thermal model. This notebook sizes the two heat-rejection paths,
**both realized as structure** so they are planned into the current geometry
rather than retrofitted (ADR-0009):

1. **ESC cold-plate** — the ESC bonds to a conductive plate on the inner
   shell wall where the EDF inflow washes it; forced convection off the
   plate carries the loss away.
2. **Vented battery bay** — inlet/outlet vents admit a through-flow that
   sweeps the battery discharge heat out of the bay.

Heat loads are **derived**: the ESC loss from the propulsion efficiency and
hover power, the battery loss as `I_hover^2 * R_pack` at the nominal pack
resistance (ADR-0014 amendment -- the same law as the mission transient); only the temperature limits and the cooling-airflow fractions
are configured (`config/thermal.yaml`).

## Inputs

- sizing loop re-run from `config/` (hover power, MTOW) *(pattern of NB2–NB6)*
- `out/fuselage.yaml` — ESC/battery-bay stations, internal diameter, mid-body length
- `config/thermal.yaml`, `config/propulsive_system_parameters.yaml`, `config/battery.yaml`

## Outputs

- `out/thermal.yaml`
- `out/thermal_paths.png`

---

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # bare-checkout fallback

from conceptual_design.notebook import nb_setup
from conceptual_design.design_point import load_handoff, solve_design_point
from conceptual_design.thermal_design import (
    ThermalParams, size_thermal, write_thermal_yaml,
    battery_pack_transient,
)
from conceptual_design.electrical_diagram import (
    ElectricalParams, compute_operating_point,
)
from conceptual_design import reports
from conceptual_design.plots import plot_thermal_paths

nb = nb_setup()
REPO_ROOT, CONFIG_PATH, OUT_PATH, FIG_DIR = (
    nb.repo_root, nb.config, nb.out, nb.fig_dir)

# Section 1 — Design Inputs

Re-run the sizing loop from `config/` (hover power + MTOW) and load the
fuselage layout for the ESC/battery-bay stations. Same pattern as NB2–NB6.

---

In [ ]:
# -- Re-run the sizing loop; heat loads derive from the hover power ------------
inp, result = solve_design_point(CONFIG_PATH)
tp = ThermalParams.from_yaml(CONFIG_PATH / "thermal.yaml")

# wiring-law operating currents (same law as NB10/NB11) -- the battery
# heat load is I_hover^2 * R_pack nominal (ADR-0014 amendment)
elec = ElectricalParams.from_yaml(CONFIG_PATH / "electrical.yaml")
op = compute_operating_point(result, inp.batt, elec)

fus = load_handoff(OUT_PATH, "fuselage")
layout = {c["name"]: c for c in fus["layout"]}

# ESC/propulsion "mounts" allocation the cold-plate must fit inside
from conceptual_design.fuselage_design import PROP_SPLIT
m_esc_alloc = PROP_SPLIT["esc"] * result.m_propulsion_kg
D_int = fus["D_fus_m"] - 2.0 * fus["t_shell_m"]

print(f"Hover power       : {result.P_hover_W:.0f} W  (MTOW {result.m_total_kg:.3f} kg)")
print(f"eta_esc           : {inp.prop.eta_esc:.2f}   "
      f"(eta_bat {inp.batt.eta_bat:.2f} mission-avg, from R_pack nominal)")
print(f"I_hover           : {op.hover_current_a:.1f} A at {op.pack_voltage_v:.1f} V")
print(f"ESC allocation    : {m_esc_alloc*1e3:.0f} g   (PROP_SPLIT esc x m_propulsion)")
print(f"Fuselage internal : D_int {D_int*1e3:.0f} mm, L_mid {fus['L_mid_m']*1e3:.0f} mm")
print(f"Battery bay        : {layout['battery']['length_m']*1e3:.0f} mm long")
print(f"Ambient / limits  : {tp.T_ambient_C:.0f} C amb, ESC<={tp.T_esc_max_C:.0f} C, "
      f"batt<={tp.T_batt_max_C:.0f} C")

# Section 2 — Size Both Thermal Paths

`thermal_design.size_thermal` computes the heat loads, the fan induced
inflow, and both paths in closed form (see the module docstring). The ESC
cold-plate is sized for its temperature limit; the battery vent for the
pack limit; each is checked against the wall area actually available in the
current geometry.

---

In [ ]:
res = size_thermal(
    P_hover_W=result.P_hover_W, eta_esc=inp.prop.eta_esc,
    I_hover_A=op.hover_current_a,
    MTOW_kg=result.m_total_kg, D_rotor_m=inp.rotor.D_rotor_m, rho=inp.env.rho,
    D_int_m=D_int, L_mid_m=fus["L_mid_m"],
    L_battery_bay_m=layout["battery"]["length_m"],
    m_esc_alloc_kg=m_esc_alloc, p=tp)

reports.print_thermal_sizing(res, tp, m_esc_alloc)

# Section 3 — Battery Pack Mission Transient (ADR-0014)

The vented-bay check above is steady-state and **air-side only**. This
section adds the lumped-capacitance pack transient from the external
battery-thermal review (C. Ucler, 2026-07): adiabatic Joule heating
$I^2 R_{pack}$ through the two vertical legs, exponential cruise cooling
in between, at the wiring-law currents and three configured
pack-resistance cases. Reported, never filtered on — same discipline
as the ESC cold-plate finding.

---

In [ ]:
trans = battery_pack_transient(
    m_bat_kg=result.m_battery_kg,
    I_hover_A=op.hover_current_a, I_cruise_A=op.cruise_current_a,
    t_hover_s=inp.mission.t_hover, t_transition_s=inp.mission.t_transition,
    t_cruise_s=inp.mission.t_cruise, V_cruise_ms=inp.mission.V_cruise,
    rho=inp.env.rho, p=tp)

reports.print_battery_transient(trans, tp)

# Section 4 — Findings

The point of a thermal pass at the conceptual stage is to catch a path that
is **not** actually free in the current geometry before the shell is frozen.

---

In [ ]:
reports.print_thermal_findings(res, tp, m_esc_alloc, trans)

# Section 5 — Visualisation

Left: the two hover heat loads. Right: the temperature each component
reaches on the available cooling wall, against its limit (bar over the
limit line = infeasible).

---

In [ ]:
plot_thermal_paths(res["esc"], res["battery"], tp, OUT_PATH / "thermal_paths.png")

# Section 6 — Output Export

`out/thermal.yaml` — consumed by the CAD notebook (cold-plate + vent
geometry) and read for reference downstream. Records both paths' margins
and the `ok` flags honestly (a marginal ESC path is reported, not masked).

---

In [ ]:
write_thermal_yaml(res, tp, OUT_PATH / "thermal.yaml", trans)
e, b = res["esc"], res["battery"]
print(f"Thermal design written -> {OUT_PATH / 'thermal.yaml'}")
print(f"  all_ok = {res['all_ok']}  (ESC {'ok' if e.ok else 'MARGINAL'}, "
      f"battery {'ok' if b.ok else 'FAIL'})")

# Section 7 — Design Summary

---

In [ ]:
reports.print_thermal_summary(res, tp, trans)